# Confounder Adjustment — European-American Cohort (GSE148812)

**Purpose:** Build the 12-dimensional confounder block (10 genetic PCs + age + gender)
for DoubleML residualization. Mirrors 02_confounders.ipynb (AA), corrected
methodology from the start (full SVD, native float64 standardization).

**Inputs:**
- `checkpoint7_snp_encoded_012.csv`, `checkpoint2_metadata_sample_filtered.csv`
- `HumanExome-12-v1-0-B.csv` manifest

**Key parameters:** identical to AA — sex-chr exclusion before PCA, monomorphic
filter (denom > 1e-8), EIGENSTRAT standardization, PCA with svd_solver='full',
10 PCs + age + gender.

**Outputs:** `confounders_X.npy`, `pca_diagnostics.csv`, `pc1_gender_admixture_summary.csv`

sex-linked exclusion:

In [1]:
import pandas as pd
import numpy as np
import re
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(probe_id):
    return re.sub(r'_\d+$', '', probe_id)

# NOTE: using checkpoint7b (relatedness-filtered), not checkpoint7
encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()

X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
print("Loaded matrix shape (SNPs x samples):", X_snp_first.shape)

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}

print("Sex-linked probes to exclude:", len(to_exclude))

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto = X_snp_first[keep_mask].T
print("X_auto shape (samples x SNPs, autosomal only):", X_auto.shape)

Loaded matrix shape (SNPs x samples): (239043, 1460)
Sex-linked probes to exclude: 5322
X_auto shape (samples x SNPs, autosomal only): (1460, 233721)


In [2]:
import numpy as np
import gc

del X_snp_first
gc.collect()

X_auto_64 = X_auto.astype(np.float64)
del X_auto
gc.collect()

p = X_auto_64.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_snp_mask = denom > 1e-8
print("Monomorphic/invalid SNPs excluded:", (~valid_snp_mask).sum())

X_standardized = (X_auto_64[:, valid_snp_mask] - 2 * p[valid_snp_mask]) / denom[valid_snp_mask]
del X_auto_64
gc.collect()

print("Standardized matrix shape:", X_standardized.shape)

probe_ids_after_autosomal = probe_id_array[keep_mask]
probe_ids_valid = probe_ids_after_autosomal[valid_snp_mask]
print("Final informative SNP count:", len(probe_ids_valid))

Monomorphic/invalid SNPs excluded: 106305
Standardized matrix shape: (1460, 127416)
Final informative SNP count: 127416


In [3]:
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"

pca_std = PCA(n_components=10, random_state=42, svd_solver='full')
pcs_std = pca_std.fit_transform(X_standardized)

print("PC scores shape:", pcs_std.shape)
print("Explained variance ratio per PC:", pca_std.explained_variance_ratio_)
print("Cumulative variance explained:", np.cumsum(pca_std.explained_variance_ratio_))

pca_diag = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(10)],
    "explained_variance_ratio": pca_std.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca_std.explained_variance_ratio_)
})
pca_diag.to_csv(os.path.join(out_dir, "pca_diagnostics_relatedness_filtered.csv"), index=False)
print("Saved.")

PC scores shape: (1460, 10)
Explained variance ratio per PC: [0.02279432 0.00591215 0.0031368  0.00311787 0.00309348 0.00308231
 0.00306748 0.00305073 0.00302753 0.00302285]
Cumulative variance explained: [0.02279432 0.02870647 0.03184327 0.03496114 0.03805462 0.04113693
 0.04420442 0.04725515 0.05028268 0.05330553]
Saved.


In [4]:
import pandas as pd
import numpy as np
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

meta_aligned = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()

age_std = ((meta_aligned["age"].astype(float) - meta_aligned["age"].astype(float).mean()) /
           meta_aligned["age"].astype(float).std()).values
gender_binary = (meta_aligned["gender"] == "Male").astype(np.float64).values

X_confounders = np.hstack([
    pcs_std,
    age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("Confounder block shape:", X_confounders.shape)

np.save(os.path.join(out_dir, "confounders_X_relatedness_filtered.npy"), X_confounders)
print("Saved.")

Confounder block shape: (1460, 12)
Saved.
